MIT License

Copyright (c) 2026 THALES

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Dataset statistics: full split census and one identified scene

This report reads released Parquet rows and declared result artifacts. It runs in the existing frozen environment: `uv run --frozen jupyter ...`; no dependency installation, training or measurement campaign is launched.

Set `dataset_path` to a local snapshot/staging root. Relative paths resolve against the discovered checkout, explicit `project_path`, or the current directory with an installed PASSAGE package. An explicit missing local input fails. Remote use requires `dataset_path=None` and a pinned `dataset_repo`/`dataset_revision`; never silently switch populations. Requested IDs remain in the requested split.

The census covers all core rows of train, calibration, validation and test. Tensor examples and classical diagnostics are finite. At every resolution missing learned receipts remain `not_run`; 256²/elevation S2–S4 evidence is required before its claims are admitted. Stored A* paths and generation times do not establish new exactness or complete-request speed claims.


In [ ]:
# Papermill overrides this cell before any imports or derived paths.
resolution = 256
project_path = None  # Optional checkout root for execution in an external staging directory.
census_limit = None  # None means full census; positive values are explicitly labelled smoke diagnostics.
dataset_path = "dataset"  # Local snapshot relative to discovered checkout; None explicitly selects pinned HF.
dataset_repo = "thalesgroup/passage"
dataset_revision = None
output_dir = "outputs/notebook-reports"
result_root = None
qualification_root = None  # Optional bound receipts; a completed census can be reused.
split = "validation"
sample_id = None  # A requested missing ID fails; no split fallback.
max_samples = 1  # Tensor gallery bound, never a census shortcut.
classical_budget = 16  # Planned validation-core diagnostic cohort; 0 disables explicitly.
max_expansions = 10000  # Finite exact-search bound, including at 4096 square.


In [ ]:
from pathlib import Path
import sys

# Discover repository root from the execution directory without changing it.
project_root = Path(project_path).resolve() if project_path else next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
if (project_root / "src/passage").is_dir():
    sys.path.insert(0, str(project_root / "src"))
# Otherwise use the installed PASSAGE package; no private checkout is required.
from passage.notebook_reports import (
    make_source, resolve_path, resolve_evidence_path, dataset_report, route_metrics_report,
    classical_diagnostics_report, evaluation_report, stored_timing_report,
    write_report, show_records, save_figure, split_inventory_report,
)
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
source = make_source(dataset_path, dataset_repo=dataset_repo,
                     dataset_revision=dataset_revision, project_root=project_root)
output_path = resolve_path(output_dir, project_root)
results_path = resolve_path(result_root, project_root)
if not isinstance(max_samples, int) or not 1 <= max_samples <= 4:
    raise ValueError("max_samples must be in 1..4 for bounded tensor galleries.")


In [ ]:
if qualification_root is not None and census_limit is None:
    from passage.qualification_reports import dataset_census_report
    report = dataset_census_report(source, resolution, resolve_evidence_path(qualification_root, project_root))
else:
    report = dataset_report(source, resolution, census_limit=census_limit)
section_statuses = {"core_census": "admitted", "qualification": report.get("qualification_status", "not_run")}
show_records(report["splits"], columns=["split", "rows", "source_tile_count", "metadata_mismatch",
    "mean_relief_m", "mean_obstacle_fraction", "qualification_status"])
display(Markdown(f"{report['population']} census; calibration remains a separate split. Metadata alone does not qualify routes or footprints."))

if report.get("statistic_scope"):
    display(Markdown(report["statistic_scope"]))


In [ ]:
report["global_split_inventory"] = split_inventory_report(source)
show_records(report["global_split_inventory"]["records"])
section_statuses["global_split_inventory"] = "admitted"


In [ ]:
gallery_row = source.get_row(resolution, split, sample_id=sample_id, samples=False)
report["task_ids"] = [gallery_row["id"]]
from passage.plotting import render_cost_comparison
figure = render_cost_comparison(gallery_row)
display(figure)
report["figures"] = save_figure(figure, output_path, f"dataset_statistics-{resolution}-{gallery_row['id']}")
plt.close(figure)
section_statuses["real_tensor_gallery"] = "admitted"


In [ ]:
report["section_statuses"] = section_statuses
report_path = write_report(report, output_path, f"dataset_statistics-{resolution}.json")
display(Markdown(f"Report receipt: `{report_path}`. Direct source fingerprints are not content hashes; source authentication and qualification are recorded separately in the bound evidence bundle when supplied."))
